# S13.1 - Full Fine-Tuning SEC Risk Classifier

This notebook fine-tunes a small transformer classifier on the Apple 2025 Form 10-K sentence classification set. It is designed as a compact teaching example: start with supervised full fine-tuning, then compare it with LoRA and QLoRA in the next notebooks.

If you run this in Colab, install the optional libraries first:

```bash
pip install -q transformers datasets evaluate accelerate scikit-learn
```

In [1]:
from pathlib import Path
import sys
import os
import dotenv

# get HF token from .env file
dotenv.load_dotenv()

# get the HF token
HF_TOKEN = os.getenv("HF_TOKEN")

def find_finance_dir(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for path in [start, *start.parents]:
        candidate = path / "help-code" / "4-nlp-finance"
        if candidate.exists():
            return candidate
        if path.name == "4-nlp-finance":
            return path
    raise FileNotFoundError("Run this notebook from the course repo or help-code/4-nlp-finance.")

FINANCE_DIR = find_finance_dir()
SRC_DIR = FINANCE_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from finance_nlp_utils import load_evaluation_csv

rows = load_evaluation_csv("apple_2025_sentence_classification.csv")
len(rows), rows[0]

(160,
 {'id': 'cls_0001',
  'sentence': 'Business Company Background The Company designs, manufactures and markets smartphones, personal computers, tablets, wearables and accessories, and sells a variety of related services.',
  'label': 'revenue_sales',
  'split': 'test',
  'label_source': 'keyword_heuristic_for_teaching'})

In [2]:
from collections import Counter

labels = sorted({row["label"] for row in rows})
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print(labels)
print(Counter(row["split"] for row in rows))
print(Counter(row["label"] for row in rows))

['cash_capital', 'legal_regulatory', 'margin_cost', 'operations_supply', 'other_disclosure', 'revenue_sales', 'risk_factor']
Counter({'train': 112, 'test': 32, 'validation': 16})
Counter({'revenue_sales': 97, 'risk_factor': 33, 'operations_supply': 12, 'cash_capital': 10, 'margin_cost': 6, 'legal_regulatory': 1, 'other_disclosure': 1})


In [3]:
from datasets import Dataset, DatasetDict

def convert(row):
    return {"text": row["sentence"], "label": label2id[row["label"]], "id": row["id"]}

splits = {}
for split in ["train", "validation", "test"]:
    splits[split] = Dataset.from_list([convert(row) for row in rows if row["split"] == split])

dataset = DatasetDict(splits)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'id'],
        num_rows: 112
    })
    validation: Dataset({
        features: ['text', 'label', 'id'],
        num_rows: 16
    })
    test: Dataset({
        features: ['text', 'label', 'id'],
        num_rows: 32
    })
})

In [4]:
from transformers import AutoTokenizer

model_name = "prajjwal1/bert-tiny"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=192)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.remove_columns(["text", "id"])
tokenized.set_format("torch")
tokenized

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {"accuracy": accuracy_score(y_true, y_pred), "macro_f1": f1, "precision": precision, "recall": recall}

args = TrainingArguments(
    output_dir=str(FINANCE_DIR / "outputs" / "sec-risk-full-ft"),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
trainer.evaluate(tokenized["test"])

In [ ]:
text = "The Company is subject to complex laws and regulations in many jurisdictions."
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=192)
inputs = {k: v.to(model.device) for k, v in inputs.items()}
pred_id = model(**inputs).logits.argmax(-1).item()
id2label[pred_id]